# Session1_Task3_short — Sales Trend Analysis

In [1]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages

s = pd.read_csv('sales_transactions_cleaned.csv')
s['date']    = pd.to_datetime(s['date'], errors='coerce')
s['month']   = s['date'].dt.to_period('M').astype(str)  # แปลงวันที่ → YYYY-MM
s['revenue'] = (s['quantity']*s['price']) - pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)

# monthly stats: revenue, transactions, avg order value
m = (s.groupby('month').agg(rev=('revenue','sum'), tx=('transaction_id','nunique'))
      .assign(aov=lambda x: x['rev']/x['tx']).sort_index().reset_index())

# top 3 months by revenue
t3 = (m.nlargest(3,'rev')[['month','rev']].reset_index(drop=True)
       .assign(rev=lambda x: x['rev'].apply(lambda v: f'${v:,.2f}')))

metrics = [('rev','tomato','Total Sales Revenue ($)'),('tx','steelblue','Number of Transactions'),('aov','seagreen','Average Order Value ($)')]

with PdfPages('Session1_SalesTrends_short.pdf') as pdf:
    for col,color,title in metrics:
        fig,ax = plt.subplots(figsize=(10,5))
        ax.plot(m['month'],m[col],marker='o',color=color,linewidth=2,markersize=5)
        ax.set(title=title,xlabel='Month'); ax.tick_params(axis='x',rotation=45)
        ax.grid(axis='y',linestyle='--',alpha=0.5)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}' if col!='tx' else f'{int(v):,}'))
        fig.tight_layout(); pdf.savefig(fig,bbox_inches='tight'); plt.close()
    fig,ax = plt.subplots(figsize=(6,2.5)); ax.axis('off')
    ax.set_title('Top 3 Months by Sales Revenue',fontsize=13,fontweight='bold',pad=16)
    tbl = ax.table(cellText=t3.values,colLabels=['Month','Total Revenue'],loc='center',cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.5,2.2)
    for (r,_),cell in tbl.get_celld().items():
        if r==0: cell.set_facecolor('steelblue'); cell.set_text_props(color='white',fontweight='bold')
        cell.set_edgecolor('lightgray')
    fig.tight_layout(); pdf.savefig(fig,bbox_inches='tight'); plt.close()

print('✅ Saved Session1_SalesTrends_short.pdf')
# จุดสังเกต: PDF 4 หน้า, แกน y มี $, top3 table มี 3 แถว

✅ Saved Session1_SalesTrends_short.pdf
